# 03 - Star Schema Design and Relationship Specification

## Purpose

This notebook converts the approved source-field, metric, category-mapping, benchmark,
and suppression specifications into a documented analytical star-schema design for
Power BI.

The notebook defines:

- The discharge-level fact-table grain
- Dimension-table grains
- Fact and dimension columns
- Source-to-model lineage
- Surrogate-key and unknown-member policies
- One-to-many relationship requirements
- Metric-field coverage
- Machine-readable schema specifications
- Business-readable schema documentation

## Inputs

Outputs from `01_data_audit.ipynb`:

- `outputs/data_audit/schema.csv`

Outputs from `02_metric_catalog_and_benchmark_design.ipynb`:

- `outputs/metric_catalog/metric_catalog.csv`
- `outputs/metric_catalog/category_mapping.csv`
- `outputs/metric_catalog/benchmark_specification.csv`
- `outputs/metric_catalog/suppression_policy.csv`
- `outputs/metric_catalog/validation_results.csv`

## Outputs

- `outputs/star_schema/table_specification.csv`
- `outputs/star_schema/column_specification.csv`
- `outputs/star_schema/source_to_model_mapping.csv`
- `outputs/star_schema/relationship_specification.csv`
- `outputs/star_schema/key_policy.csv`
- `outputs/star_schema/metric_support_matrix.csv`
- `outputs/star_schema/schema_validation_results.csv`
- `outputs/star_schema/design_standards.csv`
- `outputs/star_schema/export_manifest.csv`
- `docs/star_schema.md`

## Analytical Grain

`FactDischarge` contains one row per released inpatient discharge.

The public dataset does not contain a reliable patient identifier or durable discharge
identifier. Therefore, discharge counts must not be interpreted as unique-patient
counts.

## Scope Boundary

This notebook designs the semantic model but does not physically construct Power Query
tables, create DAX measures, build report pages, calculate peer benchmarks, or train
predictive models.

## 1. Imports

In [1]:
from pathlib import Path

import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 120)

## 2. Project Paths

In [2]:
def find_project_root(start_path):
    """Find the repository root using the existing project charter."""

    for candidate in [start_path, *start_path.parents]:
        if (candidate / "docs" / "project_charter.md").exists():
            return candidate

    raise FileNotFoundError(
        "Project root not found. Expected docs/project_charter.md "
        "in the current directory or one of its parents."
    )


PROJECT_ROOT = find_project_root(Path.cwd().resolve())

AUDIT_DIR = PROJECT_ROOT / "outputs" / "data_audit"
CATALOG_DIR = PROJECT_ROOT / "outputs" / "metric_catalog"
OUTPUT_DIR = PROJECT_ROOT / "outputs" / "star_schema"
DOCS_DIR = PROJECT_ROOT / "docs"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
DOCS_DIR.mkdir(parents=True, exist_ok=True)

print("Project root detected successfully.")
print("Project folder:", PROJECT_ROOT.name)
print("Audit Directory detected successfully.")
print("Audit Directory:", AUDIT_DIR.name)
print("Catalog Directory detected successfully.")
print("Catalog Directory:", CATALOG_DIR.name)
print("Output Directory detected successfully.")
print("Output Directory:", OUTPUT_DIR.name)

Project root detected successfully.
Project folder: 04_hospital_operations_powerbi
Audit Directory detected successfully.
Audit Directory: data_audit
Catalog Directory detected successfully.
Catalog Directory: metric_catalog
Output Directory detected successfully.
Output Directory: star_schema


### Interpretation

All paths are repository-relative. The notebook does not depend on a user-specific
Windows or OneDrive path.

## 3. Load Upstream Specifications

In [3]:
input_files = {
    "schema": AUDIT_DIR / "schema.csv",
    "metric_catalog": CATALOG_DIR / "metric_catalog.csv",
    "category_mapping": CATALOG_DIR / "category_mapping.csv",
    "benchmark_specification": (
        CATALOG_DIR / "benchmark_specification.csv"
    ),
    "suppression_policy": CATALOG_DIR / "suppression_policy.csv",
    "catalog_validation": CATALOG_DIR / "validation_results.csv",
}

missing_input_files = [
    str(file_path)
    for file_path in input_files.values()
    if not file_path.exists()
]

assert not missing_input_files, (
    "Required upstream outputs are missing:\n"
    + "\n".join(missing_input_files)
)

inputs = {
    name: pd.read_csv(file_path)
    for name, file_path in input_files.items()
}

schema = inputs["schema"]
metric_catalog = inputs["metric_catalog"]
category_mapping = inputs["category_mapping"]
benchmark_specification = inputs["benchmark_specification"]
suppression_policy = inputs["suppression_policy"]
catalog_validation = inputs["catalog_validation"]

assert "column_name" in schema.columns

available_columns = set(
    schema["column_name"].dropna().astype(str).str.strip()
)

print(f"Audited source fields: {len(available_columns)}")
print(f"Approved metrics: {len(metric_catalog)}")
print(f"Category mappings: {len(category_mapping)}")

Audited source fields: 33
Approved metrics: 32
Category mappings: 51


## 4. Upstream commit gates

In [4]:
def normalize_boolean(series):
    return (
        series.astype(str)
        .str.strip()
        .str.lower()
        .map({"true": True, "false": False})
    )


catalog_tests_passed = normalize_boolean(
    catalog_validation["passed"]
).fillna(False)

unapproved_mappings = category_mapping.loc[
    ~category_mapping["mapping_status"].eq("APPROVED")
].copy()

blank_mapping_targets = (
    category_mapping["target_group"]
    .fillna("")
    .astype(str)
    .str.strip()
    .eq("")
)

blank_mapping_rationales = (
    category_mapping["mapping_rationale"]
    .fillna("")
    .astype(str)
    .str.strip()
    .eq("")
)

assert catalog_tests_passed.all(), (
    "Notebook 02 validation results contain failures."
)

assert unapproved_mappings.empty, (
    f"{len(unapproved_mappings)} category mappings are not approved."
)

assert not blank_mapping_targets.any(), (
    "One or more approved mappings have blank target groups."
)

assert not blank_mapping_rationales.any(), (
    "One or more approved mappings have blank rationales."
)

assert metric_catalog["metric_id"].is_unique, (
    "Metric IDs are not unique."
)

required_los_benchmark_metrics = {
    "peer_expected_los_days",
    "los_actual_to_peer_expected_ratio",
    "excess_los_days_lower_bound",
}
required_cost_benchmark_metrics = {
    "peer_expected_estimated_cost",
    "estimated_cost_actual_to_peer_expected_ratio",
    "excess_estimated_cost",
}
required_benchmark_metrics = (
    required_los_benchmark_metrics
    | required_cost_benchmark_metrics
)

missing_benchmark_metrics = sorted(
    required_benchmark_metrics
    - set(metric_catalog["metric_id"])
)

assert not missing_benchmark_metrics, (
    "Notebook 02 is missing required benchmark metrics: "
    f"{missing_benchmark_metrics}"
)


def validate_primary_benchmark(benchmark_id):
    benchmark_rows = benchmark_specification.loc[
        benchmark_specification["benchmark_id"].eq(
            benchmark_id
        )
    ]

    assert len(benchmark_rows) == 1, (
        f"Notebook 02 must define exactly one {benchmark_id} benchmark."
    )

    benchmark = benchmark_rows.iloc[0]

    assert (
        benchmark["primary_peer_keys"]
        == "APR DRG Code|APR Severity of Illness Code"
    )

    assert (
        str(benchmark["fallback_peer_keys"]).strip()
        == "APR DRG Code"
    )

    assert int(benchmark["minimum_peer_n"]) == 30

    assert (
        str(
            benchmark["facility_excluded_from_own_peer"]
        )
        .strip()
        .lower()
        == "true"
    )

    assert benchmark["peer_statistic"] == "Arithmetic mean"
    assert benchmark["status"] == "APPROVED_BASELINE"


validate_primary_benchmark("LOS_PEER_MEAN_PRIMARY")
validate_primary_benchmark("COST_PEER_MEAN_PRIMARY")

print("Notebook 02 commit gates passed.")

Notebook 02 commit gates passed.


## 5. Modeling Standards

In [5]:
design_standards = pd.DataFrame(
    [
        {
            "standard_id": "FACT_GRAIN",
            "decision": (
                "FactDischarge contains one row per released inpatient "
                "discharge."
            ),
            "rationale": (
                "This matches the audited source grain."
            ),
        },
        {
            "standard_id": "NO_PATIENT_KEY",
            "decision": (
                "The model will not create or imply a unique-patient key."
            ),
            "rationale": (
                "The public file does not support patient-level linkage."
            ),
        },
        {
            "standard_id": "STAR_SCHEMA",
            "decision": (
                "Dimensions relate directly to FactDischarge."
            ),
            "rationale": (
                "This avoids snowflaking and ambiguous filtering."
            ),
        },
        {
            "standard_id": "SERVICE_GRAIN",
            "decision": (
                "DimService uses APR-DRG Code and APR MDC Code "
                "as a composite natural key."
            ),
            "rationale": (
                "Physical validation showed that APR-DRG Code alone "
                "does not uniquely determine APR MDC. The composite "
                "APR-DRG and APR MDC key resolves the observed "
                "one-to-many mapping while preserving both classifications."
            ),
        },
        {
        "standard_id": "HOSPITAL_GRAIN",
        "decision": (
            "DimHospital uses Permanent Facility Id as its natural key. "
            "Operating Certificate Number remains staging-only."
        ),
        "rationale": (
            "Physical validation identified a Permanent Facility Id "
            "with multiple non-null operating certificate numbers. "
            "PFI remains the facility-level identifier, while operating "
            "certificate number is not treated as a stable descriptive "
            "attribute."
        ),
        },

        {
            "standard_id": "RELATIONSHIP_DIRECTION",
            "decision": (
                "Relationships are active, one-to-many, and "
                "single-directional from dimension to fact."
            ),
            "rationale": (
                "This provides predictable Power BI filter behavior."
            ),
        },
        {
            "standard_id": "SURROGATE_KEYS",
            "decision": (
                "Whole-number surrogate keys are used for model "
                "relationships."
            ),
            "rationale": (
                "Integer keys support model compression and separate "
                "relationships from released business identifiers."
            ),
        },
        {
            "standard_id": "UNKNOWN_MEMBER",
            "decision": (
                "Surrogate key 0 is reserved for Unknown / Not Available."
            ),
            "rationale": (
                "Missing dimension values must not cause discharge rows "
                "to be dropped."
            ),
        },
        {
            "standard_id": "TIME_GRAIN",
            "decision": (
                "DimDate initially contains one row per available "
                "discharge year."
            ),
            "rationale": (
                "The audited source supports annual—not monthly—analysis."
            ),
        },
        {
            "standard_id": "PRIMARY_PAYER",
            "decision": (
                "The initial semantic model uses Payment Typology 1."
            ),
            "rationale": (
                "Secondary and tertiary payer fields would require "
                "role-playing or bridge-table design not required by "
                "the approved metrics."
            ),
        },
        {
            "standard_id": "PATIENT_GEOGRAPHY",
            "decision": (
                "DimGeography represents released patient ZIP3 geography."
            ),
            "rationale": (
                "Hospital county and service area remain attributes of "
                "DimHospital."
            ),
        },
        {
            "standard_id": "POST_OUTCOME_CONTEXT",
            "decision": (
                "Patient disposition may be used for descriptive slicing "
                "but not as an input to the primary expected-LOS or "
                "expected-cost benchmark."
            ),
            "rationale": (
                "Disposition may reflect events occurring during or after "
                "the hospitalization."
            ),
        },
        {
            "standard_id": "PREDICTION_OUTPUT_EXTENSION",
            "decision": (
                "Validated model predictions will be stored separately from "
                "descriptive peer-benchmark values and versioned during modeling."
            ),
            "rationale": (
                "Peer-expected LOS must not be confused with model-predicted LOS, "
                "and model version and scoring date must remain traceable."
            ),
        },
        {
            "standard_id": "BENCHMARK_COLUMNS",
            "decision": (
                "Peer-expected LOS and estimated-cost fields are planned "
                "in FactDischarge but remain deferred until benchmark computation."
            ),
            "rationale": (
                "Notebook 03 defines storage requirements without "
                "claiming that the benchmark has been calculated."
            ),
        },
    ]
)

design_standards

,standard_id,decision,rationale
0,FACT_GRAIN,FactDischarge contains one row per released inpatient discharge.,This matches the audited source grain.
1,NO_PATIENT_KEY,The model will not create or imply a unique-patient key.,The public file does not support patient-level linkage.
2,STAR_SCHEMA,Dimensions relate directly to FactDischarge.,This avoids snowflaking and ambiguous filtering.
3,SERVICE_GRAIN,DimService uses APR-DRG Code and APR MDC Code as a composite natural key.,Physical validation showed that APR-DRG Code alone does not uniquely determine APR MDC. The composite APR-DRG and AP...
4,HOSPITAL_GRAIN,DimHospital uses Permanent Facility Id as its natural key. Operating Certificate Number remains staging-only.,Physical validation identified a Permanent Facility Id with multiple non-null operating certificate numbers. PFI rem...
5,RELATIONSHIP_DIRECTION,"Relationships are active, one-to-many, and single-directional from dimension to fact.",This provides predictable Power BI filter behavior.
6,SURROGATE_KEYS,Whole-number surrogate keys are used for model relationships.,Integer keys support model compression and separate relationships from released business identifiers.
7,UNKNOWN_MEMBER,Surrogate key 0 is reserved for Unknown / Not Available.,Missing dimension values must not cause discharge rows to be dropped.
8,TIME_GRAIN,DimDate initially contains one row per available discharge year.,The audited source supports annual—not monthly—analysis.
9,PRIMARY_PAYER,The initial semantic model uses Payment Typology 1.,Secondary and tertiary payer fields would require role-playing or bridge-table design not required by the approved m...


## 6. Table Specification

In [6]:
table_specification = pd.DataFrame(
    [
        {
            "table_name": "FactDischarge",
            "table_type": "Fact",
            "grain": "One row per released inpatient discharge",
            "primary_key": "",
            "natural_key": "",
            "unknown_member_required": False,
            "build_session": "Session 5",
            "business_role": (
                "Stores discharge-level foreign keys, LOS, financial "
                "amounts, validity flags, and later benchmark values."
            ),
            "important_caveat": (
                "No durable public discharge or patient identifier exists."
            ),
        },
        {
            "table_name": "DimHospital",
            "table_type": "Dimension",
            "grain": "One row per permanent facility identifier",
            "primary_key": "hospital_key",
            "natural_key": "permanent_facility_id",
            "unknown_member_required": True,
            "build_session": "Session 5",
            "business_role": "Hospital and released hospital geography.",
            "important_caveat": (
                "Future multi-year loads must test whether hospital "
                "attributes change over time."
            ),
        },
        {
            "table_name": "DimDate",
            "table_type": "Dimension",
            "grain": "One row per available discharge year",
            "primary_key": "date_key",
            "natural_key": "discharge_year",
            "unknown_member_required": True,
            "build_session": "Session 5",
            "business_role": "Annual reporting context.",
            "important_caveat": "This does not support monthly trends.",
        },
        {
            "table_name": "DimService",
            "table_type": "Dimension",
            "grain": (
                "One row per APR-DRG code and APR MDC code combination"
            ),
            "primary_key": "service_key",
            "natural_key": "apr_drg_code|apr_mdc_code",
            "unknown_member_required": True,
            "build_session": "Session 5",
            "business_role": (
                "APR-DRG, MDC, and medical/surgical service classification."
            ),
            "important_caveat": (
                "APR-DRG Code alone does not uniquely determine APR MDC "
                "in the audited source. The validated composite natural key "
                "is APR-DRG Code plus APR MDC Code."
            ),
        },
        {
            "table_name": "DimCaseMix",
            "table_type": "Dimension",
            "grain": (
                "One row per APR severity and mortality-risk combination"
            ),
            "primary_key": "case_mix_key",
            "natural_key": (
                "apr_severity_code|apr_mortality_risk"
            ),
            "unknown_member_required": True,
            "build_session": "Session 5",
            "business_role": (
                "Released severity and mortality-risk classifications."
            ),
            "important_caveat": (
                "These classifications provide context but do not capture "
                "all case-complexity differences."
            ),
        },
        {
            "table_name": "DimDiagnosis",
            "table_type": "Dimension",
            "grain": "One row per released CCSR diagnosis code",
            "primary_key": "diagnosis_key",
            "natural_key": "ccsr_diagnosis_code",
            "unknown_member_required": True,
            "build_session": "Session 5",
            "business_role": "Released diagnosis classification.",
            "important_caveat": (
                "The public classification is not a complete clinical history."
            ),
        },
        {
            "table_name": "DimProcedure",
            "table_type": "Dimension",
            "grain": "One row per released CCSR procedure code",
            "primary_key": "procedure_key",
            "natural_key": "ccsr_procedure_code",
            "unknown_member_required": True,
            "build_session": "Session 5",
            "business_role": "Released procedure classification.",
            "important_caveat": (
                "Missing procedure categories remain represented."
            ),
        },
        {
            "table_name": "DimPatientSegment",
            "table_type": "Dimension",
            "grain": (
                "One row per age-group, gender, race, and ethnicity "
                "combination"
            ),
            "primary_key": "patient_segment_key",
            "natural_key": "age_group|gender|race|ethnicity",
            "unknown_member_required": True,
            "build_session": "Session 5",
            "business_role": "Released demographic segmentation.",
            "important_caveat": (
                "This is a demographic segment—not a unique patient."
            ),
        },
        {
            "table_name": "DimGeography",
            "table_type": "Dimension",
            "grain": "One row per released patient ZIP3 value",
            "primary_key": "geography_key",
            "natural_key": "patient_zip3",
            "unknown_member_required": True,
            "build_session": "Session 5",
            "business_role": "Coarse patient-residence geography.",
            "important_caveat": (
                "ZIP3 does not identify an exact residence."
            ),
        },
        {
            "table_name": "DimPayer",
            "table_type": "Dimension",
            "grain": "One row per approved primary-payer group",
            "primary_key": "payer_key",
            "natural_key": "payer_group",
            "unknown_member_required": True,
            "build_session": "Session 5",
            "business_role": "Approved primary-payer classification.",
            "important_caveat": (
                "The initial model excludes secondary and tertiary payer roles."
            ),
        },
        {
            "table_name": "DimAdmissionContext",
            "table_type": "Dimension",
            "grain": (
                "One row per admission-type, disposition, and ED-indicator "
                "combination"
            ),
            "primary_key": "admission_context_key",
            "natural_key": (
                "admission_type_group|disposition_group|ed_indicator_group"
            ),
            "unknown_member_required": True,
            "build_session": "Session 5",
            "business_role": (
                "Admission-pathway and discharge-disposition context."
            ),
            "important_caveat": (
                "Patient disposition must not be treated as an "
                "admission-time predictor."
            ),
        },
        {
            "table_name": "Measures",
            "table_type": "Measure table",
            "grain": "One technical placeholder row",
            "primary_key": "",
            "natural_key": "",
            "unknown_member_required": False,
            "build_session": "Session 6",
            "business_role": "Dedicated location for governed DAX measures.",
            "important_caveat": (
                "This table remains disconnected from the star schema."
            ),
        },
    ]
)

table_specification

,table_name,table_type,grain,primary_key,natural_key,unknown_member_required,build_session,business_role,important_caveat
0,FactDischarge,Fact,One row per released inpatient discharge,,,False,Session 5,"Stores discharge-level foreign keys, LOS, financial amounts, validity flags, and later benchmark values.",No durable public discharge or patient identifier exists.
1,DimHospital,Dimension,One row per permanent facility identifier,hospital_key,permanent_facility_id,True,Session 5,Hospital and released hospital geography.,Future multi-year loads must test whether hospital attributes change over time.
2,DimDate,Dimension,One row per available discharge year,date_key,discharge_year,True,Session 5,Annual reporting context.,This does not support monthly trends.
3,DimService,Dimension,One row per APR-DRG code and APR MDC code combination,service_key,apr_drg_code|apr_mdc_code,True,Session 5,"APR-DRG, MDC, and medical/surgical service classification.",APR-DRG Code alone does not uniquely determine APR MDC in the audited source. The validated composite natural key is...
4,DimCaseMix,Dimension,One row per APR severity and mortality-risk combination,case_mix_key,apr_severity_code|apr_mortality_risk,True,Session 5,Released severity and mortality-risk classifications.,These classifications provide context but do not capture all case-complexity differences.
5,DimDiagnosis,Dimension,One row per released CCSR diagnosis code,diagnosis_key,ccsr_diagnosis_code,True,Session 5,Released diagnosis classification.,The public classification is not a complete clinical history.
6,DimProcedure,Dimension,One row per released CCSR procedure code,procedure_key,ccsr_procedure_code,True,Session 5,Released procedure classification.,Missing procedure categories remain represented.
7,DimPatientSegment,Dimension,"One row per age-group, gender, race, and ethnicity combination",patient_segment_key,age_group|gender|race|ethnicity,True,Session 5,Released demographic segmentation.,This is a demographic segment—not a unique patient.
8,DimGeography,Dimension,One row per released patient ZIP3 value,geography_key,patient_zip3,True,Session 5,Coarse patient-residence geography.,ZIP3 does not identify an exact residence.
9,DimPayer,Dimension,One row per approved primary-payer group,payer_key,payer_group,True,Session 5,Approved primary-payer classification.,The initial model excludes secondary and tertiary payer roles.


## 7. Column Specifications

In [7]:
column_columns = [
    "table_name",
    "column_name",
    "data_type",
    "column_role",
    "source_fields",
    "default_summarization",
    "hidden_in_report",
    "description",
    "status",
]


def define_column(
    table_name,
    column_name,
    data_type,
    column_role,
    source_fields,
    default_summarization,
    hidden_in_report,
    description,
    status="PLANNED",
):
    return {
        "table_name": table_name,
        "column_name": column_name,
        "data_type": data_type,
        "column_role": column_role,
        "source_fields": source_fields,
        "default_summarization": default_summarization,
        "hidden_in_report": hidden_in_report,
        "description": description,
        "status": status,
    }


column_rows = []

### Deferred Model-Prediction Extension

The current schema supports descriptive metrics and peer benchmarks. It does
not yet represent validated machine-learning predictions.

The modeling phase must define a versioned prediction artifact containing:

- A technical source-record key scoped to the source snapshot
- Source snapshot or file identifier
- Model version
- Scoring date and time
- Model-predicted LOS
- Prediction status or coverage flag
- Missing-prediction reason
- Optional P50 or P90 prediction when validated

The technical record key must not be described as a patient identifier or a
durable longitudinal discharge identifier.

`peer_expected_los_days` and `model_predicted_los_days` must remain separate
concepts.

### Fact foreign keys

In [8]:
fact_foreign_keys = [
    (
        "hospital_key",
        "Permanent Facility Id",
        "Resolves each discharge to DimHospital.",
    ),
    (
        "date_key",
        "Discharge Year",
        "Resolves each discharge to annual DimDate.",
    ),
    (
        "service_key",
        "APR DRG Code|APR MDC Code",
        (
            "Resolves each discharge to DimService using the "
            "APR-DRG and APR MDC composite natural key."
        ),
    ),
    (
        "case_mix_key",
        "APR Severity of Illness Code|APR Risk of Mortality",
        "Resolves each discharge to DimCaseMix.",
    ),
    (
        "diagnosis_key",
        "CCSR Diagnosis Code",
        "Resolves each discharge to DimDiagnosis.",
    ),
    (
        "procedure_key",
        "CCSR Procedure Code",
        "Resolves each discharge to DimProcedure.",
    ),
    (
        "patient_segment_key",
        "Age Group|Gender|Race|Ethnicity",
        "Resolves each discharge to DimPatientSegment.",
    ),
    (
        "geography_key",
        "Zip Code - 3 digits",
        "Resolves each discharge to DimGeography.",
    ),
    (
        "payer_key",
        "Payment Typology 1",
        "Resolves each discharge to DimPayer.",
    ),
    (
        "admission_context_key",
        (
            "Type of Admission|Patient Disposition|"
            "Emergency Department Indicator"
        ),
        "Resolves each discharge to DimAdmissionContext.",
    ),
]

for column_name, source_fields, description in fact_foreign_keys:
    column_rows.append(
        define_column(
            table_name="FactDischarge",
            column_name=column_name,
            data_type="Whole number",
            column_role="Foreign key",
            source_fields=source_fields,
            default_summarization="Do not summarize",
            hidden_in_report=True,
            description=description,
        )
    )

### Fact values and flags

In [9]:
fact_value_columns = [
    (
        "source_record_key",
        "Whole number",
        "Technical row key",
        "",
        "Do not summarize",
        (
            "Unique row ordinal scoped to the audited source snapshot. "
            "It is not a patient identifier or durable discharge identifier."
        ),
        "PLANNED",
    ),
    (
        "los_days_lower_bound",
        "Whole number",
        "Additive value",
        "Length of Stay",
        "Sum",
        "Observable LOS lower bound after representing 120+ as 120.",
        "PLANNED",
    ),
    (
        "is_top_coded_los",
        "Whole number",
        "Validity flag",
        "Length of Stay",
        "Sum",
        "Equals 1 when the released LOS is top-coded; otherwise 0.",
        "PLANNED",
    ),
    (
        "is_valid_los",
        "Whole number",
        "Validity flag",
        "Length of Stay",
        "Sum",
        "Equals 1 when LOS is valid for LOS calculations.",
        "PLANNED",
    ),
    (
        "total_charges",
        "Fixed decimal number",
        "Additive value",
        "Total Charges",
        "Sum",
        "Valid positive released total charges.",
        "PLANNED",
    ),
    (
        "is_valid_charge",
        "Whole number",
        "Validity flag",
        "Total Charges",
        "Sum",
        "Equals 1 when Total Charges is numeric and positive.",
        "PLANNED",
    ),
    (
        "total_costs",
        "Fixed decimal number",
        "Additive value",
        "Total Costs",
        "Sum",
        "Valid positive released estimated total costs.",
        "PLANNED",
    ),
    (
        "is_valid_cost",
        "Whole number",
        "Validity flag",
        "Total Costs",
        "Sum",
        "Equals 1 when Total Costs is numeric and positive.",
        "PLANNED",
    ),
    (
        "is_paired_financial_valid",
        "Whole number",
        "Validity flag",
        "Total Charges|Total Costs",
        "Sum",
        (
            "Equals 1 when charges and costs are both valid "
            "on the same discharge."
        ),
        "PLANNED",
    ),
    (
        "peer_expected_los_days",
        "Decimal number",
        "Additive benchmark value",
        (
            "Permanent Facility Id|APR DRG Code|"
            "APR Severity of Illness Code|Length of Stay"
        ),
        "Sum",
        (
            "Leave-one-facility-out expected LOS lower bound. "
            "Not populated by this notebook."
        ),
        "DEFERRED_BENCHMARK_BUILD",
    ),
    (
        "los_peer_comparison_n",
        "Whole number",
        "Benchmark diagnostic",
        (
            "Permanent Facility Id|APR DRG Code|"
            "APR Severity of Illness Code|Length of Stay"
        ),
        "Do not summarize",
        (
            "Comparison-discharge count at the selected peer level "
            "after excluding the benchmarked facility."
        ),
        "DEFERRED_BENCHMARK_BUILD",
    ),
    (
    "peer_expected_estimated_cost",
    "Fixed decimal number",
    "Additive benchmark value",
    (
        "Permanent Facility Id|APR DRG Code|"
        "APR Severity of Illness Code|Total Costs"
    ),
    "Sum",
    (
        "Leave-one-facility-out peer-expected estimated cost. "
        "Not populated by this notebook."
    ),
    "DEFERRED_BENCHMARK_BUILD",
    ),
    (
        "los_peer_benchmark_level",
        "Text",
        "Benchmark diagnostic",
        (
            "Permanent Facility Id|APR DRG Code|"
            "APR Severity of Illness Code|Length of Stay"
        ),
        "Do not summarize",
        (
            "Indicates primary, APR-DRG fallback, or unavailable "
            "peer benchmark."
        ),
        "DEFERRED_BENCHMARK_BUILD",
    ),
    (
    "cost_peer_comparison_n",
    "Whole number",
    "Benchmark diagnostic",
    (
        "Permanent Facility Id|APR DRG Code|"
        "APR Severity of Illness Code|Total Costs"
    ),
    "Do not summarize",
    (
        "Comparison-discharge count for the selected cost peer level "
        "after excluding the benchmarked facility."
    ),
    "DEFERRED_BENCHMARK_BUILD",
    ),
    (
        "cost_peer_benchmark_level",
        "Text",
        "Benchmark diagnostic",
        (
            "Permanent Facility Id|APR DRG Code|"
            "APR Severity of Illness Code|Total Costs"
        ),
        "Do not summarize",
        (
            "Indicates primary, APR-DRG fallback, or unavailable "
            "estimated-cost benchmark."
        ),
        "DEFERRED_BENCHMARK_BUILD",
    ),
]

for (
    column_name,
    data_type,
    column_role,
    source_fields,
    summarization,
    description,
    status,
) in fact_value_columns:
    column_rows.append(
        define_column(
            table_name="FactDischarge",
            column_name=column_name,
            data_type=data_type,
            column_role=column_role,
            source_fields=source_fields,
            default_summarization=summarization,
            hidden_in_report=True,
            description=description,
            status=status,
        )
    )

### Dimension Columns

In [10]:
dimension_columns = {
    "DimHospital": [
        (
            "hospital_key",
            "Whole number",
            "Primary key",
            "Permanent Facility Id",
            True,
            "Model-generated hospital surrogate key.",
        ),
        (
            "permanent_facility_id",
            "Text",
            "Natural key",
            "Permanent Facility Id",
            False,
            "Released permanent facility identifier.",
        ),
        (
            "facility_name",
            "Text",
            "Attribute",
            "Facility Name",
            False,
            "Released facility name.",
        ),
        (
            "hospital_service_area",
            "Text",
            "Attribute",
            "Hospital Service Area",
            False,
            "Hospital service area.",
        ),
        (
            "hospital_county",
            "Text",
            "Attribute",
            "Hospital County",
            False,
            "Hospital county.",
        ),
    ],
    "DimDate": [
        (
            "date_key",
            "Whole number",
            "Primary key",
            "Discharge Year",
            True,
            "Annual model key derived from discharge year.",
        ),
        (
            "discharge_year",
            "Whole number",
            "Natural key",
            "Discharge Year",
            False,
            "Released discharge year.",
        ),
        (
            "year_label",
            "Text",
            "Attribute",
            "Discharge Year",
            False,
            "Business-readable year label.",
        ),
    ],
    "DimService": [
        (
            "service_key",
            "Whole number",
            "Primary key",
            "APR DRG Code|APR MDC Code",
            True,
            (
                "Model-generated service surrogate key derived from "
                "the APR-DRG and APR MDC composite natural key."
            ),
        ),
        (
            "apr_drg_code",
            "Text",
            "Natural key component",
            "APR DRG Code",
            False,
            "APR-DRG component of the DimService composite natural key.",
        ),
        (
            "apr_drg_description",
            "Text",
            "Attribute",
            "APR DRG Description",
            False,
            "APR-DRG description.",
        ),
        (
            "apr_mdc_code",
            "Text",
            "Natural key component",
            "APR MDC Code",
            False,
            (
                "APR MDC component of the DimService composite natural key."
            ),
        ),
        (
            "apr_mdc_description",
            "Text",
            "Attribute",
            "APR MDC Description",
            False,
            "APR major diagnostic category description.",
        ),
        (
            "medical_surgical_classification",
            "Text",
            "Attribute",
            "APR Medical Surgical Description",
            False,
            "Released APR medical/surgical classification.",
        ),
    ],
    "DimCaseMix": [
        (
            "case_mix_key",
            "Whole number",
            "Primary key",
            (
                "APR Severity of Illness Code|"
                "APR Risk of Mortality"
            ),
            True,
            "Model-generated case-mix surrogate key.",
        ),
        (
            "apr_severity_code",
            "Whole number",
            "Natural-key component",
            "APR Severity of Illness Code",
            False,
            (
                "Validated APR severity code from 1 through 4; "
                "also used as the descriptive severity index."
            ),
        ),
        (
            "apr_severity_description",
            "Text",
            "Attribute",
            "APR Severity of Illness Description",
            False,
            "APR severity description.",
        ),
        (
            "apr_mortality_risk",
            "Text",
            "Natural-key component",
            "APR Risk of Mortality",
            False,
            "APR mortality-risk category.",
        ),
    ],
    "DimDiagnosis": [
        (
            "diagnosis_key",
            "Whole number",
            "Primary key",
            "CCSR Diagnosis Code",
            True,
            "Model-generated diagnosis surrogate key.",
        ),
        (
            "ccsr_diagnosis_code",
            "Text",
            "Natural key",
            "CCSR Diagnosis Code",
            False,
            "Released CCSR diagnosis code.",
        ),
        (
            "ccsr_diagnosis_description",
            "Text",
            "Attribute",
            "CCSR Diagnosis Description",
            False,
            "Released CCSR diagnosis description.",
        ),
    ],
    "DimProcedure": [
        (
            "procedure_key",
            "Whole number",
            "Primary key",
            "CCSR Procedure Code",
            True,
            "Model-generated procedure surrogate key.",
        ),
        (
            "ccsr_procedure_code",
            "Text",
            "Natural key",
            "CCSR Procedure Code",
            False,
            "Released CCSR procedure code.",
        ),
        (
            "ccsr_procedure_description",
            "Text",
            "Attribute",
            "CCSR Procedure Description",
            False,
            "Released CCSR procedure description.",
        ),
    ],
    "DimPatientSegment": [
        (
            "patient_segment_key",
            "Whole number",
            "Primary key",
            "Age Group|Gender|Race|Ethnicity",
            True,
            "Model-generated demographic-segment key.",
        ),
        (
            "age_group",
            "Text",
            "Natural-key component",
            "Age Group",
            False,
            "Released age group.",
        ),
        (
            "gender",
            "Text",
            "Natural-key component",
            "Gender",
            False,
            "Released gender category.",
        ),
        (
            "race",
            "Text",
            "Natural-key component",
            "Race",
            False,
            "Released race category.",
        ),
        (
            "ethnicity",
            "Text",
            "Natural-key component",
            "Ethnicity",
            False,
            "Released ethnicity category.",
        ),
    ],
    "DimGeography": [
        (
            "geography_key",
            "Whole number",
            "Primary key",
            "Zip Code - 3 digits",
            True,
            "Model-generated patient-geography key.",
        ),
        (
            "patient_zip3",
            "Text",
            "Natural key",
            "Zip Code - 3 digits",
            False,
            "Released three-digit patient ZIP geography.",
        ),
    ],
    "DimPayer": [
        (
            "payer_key",
            "Whole number",
            "Primary key",
            "Payment Typology 1",
            True,
            "Model-generated primary-payer key.",
        ),
        (
            "payer_group",
            "Text",
            "Natural key",
            "Payment Typology 1",
            False,
            "Approved mapped primary-payer group.",
        ),
    ],
    "DimAdmissionContext": [
        (
            "admission_context_key",
            "Whole number",
            "Primary key",
            (
                "Type of Admission|Patient Disposition|"
                "Emergency Department Indicator"
            ),
            True,
            "Model-generated admission-context key.",
        ),
        (
            "admission_type_group",
            "Text",
            "Natural-key component",
            "Type of Admission",
            False,
            "Approved mapped admission-type group.",
        ),
        (
            "disposition_group",
            "Text",
            "Natural-key component",
            "Patient Disposition",
            False,
            "Approved mapped disposition group.",
        ),
        (
            "ed_indicator_group",
            "Text",
            "Natural-key component",
            "Emergency Department Indicator",
            False,
            "Approved mapped ED-indicator group.",
        ),
    ],
}

for table_name, specifications in dimension_columns.items():
    for (
        column_name,
        data_type,
        column_role,
        source_fields,
        hidden,
        description,
    ) in specifications:
        column_rows.append(
            define_column(
                table_name=table_name,
                column_name=column_name,
                data_type=data_type,
                column_role=column_role,
                source_fields=source_fields,
                default_summarization="Do not summarize",
                hidden_in_report=hidden,
                description=description,
            )
        )

column_rows.append(
    define_column(
        table_name="Measures",
        column_name="_measure_table_marker",
        data_type="Whole number",
        column_role="Technical placeholder",
        source_fields="",
        default_summarization="Do not summarize",
        hidden_in_report=True,
        description=(
            "Placeholder column used only to host governed measures."
        ),
        status="SESSION_6",
    )
)

column_specification = pd.DataFrame(
    column_rows,
    columns=column_columns,
)

column_specification

,table_name,column_name,data_type,column_role,source_fields,default_summarization,hidden_in_report,description,status
0,FactDischarge,hospital_key,Whole number,Foreign key,Permanent Facility Id,Do not summarize,True,Resolves each discharge to DimHospital.,PLANNED
1,FactDischarge,date_key,Whole number,Foreign key,Discharge Year,Do not summarize,True,Resolves each discharge to annual DimDate.,PLANNED
2,FactDischarge,service_key,Whole number,Foreign key,APR DRG Code|APR MDC Code,Do not summarize,True,Resolves each discharge to DimService using the APR-DRG and APR MDC composite natural key.,PLANNED
3,FactDischarge,case_mix_key,Whole number,Foreign key,APR Severity of Illness Code|APR Risk of Mortality,Do not summarize,True,Resolves each discharge to DimCaseMix.,PLANNED
4,FactDischarge,diagnosis_key,Whole number,Foreign key,CCSR Diagnosis Code,Do not summarize,True,Resolves each discharge to DimDiagnosis.,PLANNED
...,...,...,...,...,...,...,...,...,...
58,DimAdmissionContext,admission_context_key,Whole number,Primary key,Type of Admission|Patient Disposition|Emergency Department Indicator,Do not summarize,True,Model-generated admission-context key.,PLANNED
59,DimAdmissionContext,admission_type_group,Text,Natural-key component,Type of Admission,Do not summarize,False,Approved mapped admission-type group.,PLANNED
60,DimAdmissionContext,disposition_group,Text,Natural-key component,Patient Disposition,Do not summarize,False,Approved mapped disposition group.,PLANNED
61,DimAdmissionContext,ed_indicator_group,Text,Natural-key component,Emergency Department Indicator,Do not summarize,False,Approved mapped ED-indicator group.,PLANNED


## 8. Source-to-model mapping

In [11]:
source_mapping_rows = [
    (
        "Hospital Service Area",
        "DimHospital",
        "hospital_service_area",
        "INCLUDE",
        "Trim text and map nulls to Unknown.",
    ),
    (
        "Hospital County",
        "DimHospital",
        "hospital_county",
        "INCLUDE",
        "Trim text and map nulls to Unknown.",
    ),
    (
        "Operating Certificate Number",
        "Staging",
        "",
        "STAGING_ONLY",
        (
            "Retained for source validation and lineage only. "
            "Not promoted to DimHospital because one Permanent "
            "Facility Id may have multiple operating certificate "
            "numbers in the audited source."
        ),
    ),
    (
        "Permanent Facility Id",
        "DimHospital",
        "permanent_facility_id",
        "INCLUDE",
        "Retain natural identifier and derive hospital_key.",
    ),
    (
        "Facility Name",
        "DimHospital",
        "facility_name",
        "INCLUDE",
        "Trim text; do not use as the hospital key.",
    ),
    (
        "Age Group",
        "DimPatientSegment",
        "age_group",
        "INCLUDE",
        "Apply the approved category mapping.",
    ),
    (
        "Zip Code - 3 digits",
        "DimGeography",
        "patient_zip3",
        "INCLUDE",
        "Retain as text and preserve masked/unknown values.",
    ),
    (
        "Gender",
        "DimPatientSegment",
        "gender",
        "INCLUDE",
        "Trim and preserve released categories.",
    ),
    (
        "Race",
        "DimPatientSegment",
        "race",
        "INCLUDE",
        "Trim and preserve released categories.",
    ),
    (
        "Ethnicity",
        "DimPatientSegment",
        "ethnicity",
        "INCLUDE",
        "Trim and preserve released categories.",
    ),
    (
        "Length of Stay",
        "FactDischarge",
        "los_days_lower_bound",
        "INCLUDE",
        "Parse valid values; represent 120+ as 120 and create flags.",
    ),
    (
        "Type of Admission",
        "DimAdmissionContext",
        "admission_type_group",
        "INCLUDE",
        "Apply the approved category mapping.",
    ),
    (
        "Patient Disposition",
        "DimAdmissionContext",
        "disposition_group",
        "INCLUDE",
        "Apply approved mapping; descriptive use only.",
    ),
    (
        "Discharge Year",
        "DimDate",
        "discharge_year",
        "INCLUDE",
        "Convert to whole number and derive date_key.",
    ),
    (
        "CCSR Diagnosis Code",
        "DimDiagnosis",
        "ccsr_diagnosis_code",
        "INCLUDE",
        "Retain as text and derive diagnosis_key.",
    ),
    (
        "CCSR Diagnosis Description",
        "DimDiagnosis",
        "ccsr_diagnosis_description",
        "INCLUDE",
        "Trim text.",
    ),
    (
        "CCSR Procedure Code",
        "DimProcedure",
        "ccsr_procedure_code",
        "INCLUDE",
        "Retain as text and derive procedure_key.",
    ),
    (
        "CCSR Procedure Description",
        "DimProcedure",
        "ccsr_procedure_description",
        "INCLUDE",
        "Trim text.",
    ),
    (
        "APR DRG Code",
        "DimService",
        "apr_drg_code",
        "INCLUDE",
        (
            "Retain as text and combine with APR MDC Code "
            "to derive service_key."
        ),
    ),
    (
        "APR DRG Description",
        "DimService",
        "apr_drg_description",
        "INCLUDE",
        "Trim text.",
    ),
    (
        "APR MDC Code",
        "DimService",
        "apr_mdc_code",
        "INCLUDE",
        (
            "Retain as text and combine with APR DRG Code "
            "to derive service_key."
        ),
    ),
    (
        "APR MDC Description",
        "DimService",
        "apr_mdc_description",
        "INCLUDE",
        "Trim text.",
    ),
    (
        "APR Severity of Illness Code",
        "DimCaseMix",
        "apr_severity_code",
        "INCLUDE",
        (
            "Parse as a whole number from 1 through 4, validate "
            "against the severity description, and derive case_mix_key."
        ),
    ),
    (
        "APR Severity of Illness Description",
        "DimCaseMix",
        "apr_severity_description",
        "INCLUDE",
        "Apply the approved category mapping.",
    ),
    (
        "APR Risk of Mortality",
        "DimCaseMix",
        "apr_mortality_risk",
        "INCLUDE",
        "Apply the approved category mapping.",
    ),
    (
        "APR Medical Surgical Description",
        "DimService",
        "medical_surgical_classification",
        "INCLUDE",
        "Trim and preserve released categories.",
    ),
    (
        "Payment Typology 1",
        "DimPayer",
        "payer_group",
        "INCLUDE",
        "Apply the approved primary-payer mapping.",
    ),
    (
        "Payment Typology 2",
        "",
        "",
        "STAGING_ONLY",
        (
            "Retain in staging; secondary-payer modeling requires a "
            "separate role-playing or bridge-table design."
        ),
    ),
    (
        "Payment Typology 3",
        "",
        "",
        "STAGING_ONLY",
        (
            "Retain in staging; tertiary-payer modeling requires a "
            "separate role-playing or bridge-table design."
        ),
    ),
    (
        "Birth Weight",
        "",
        "",
        "STAGING_ONLY",
        (
            "Retain in staging until a validated newborn-specific metric "
            "or segment is approved."
        ),
    ),
    (
        "Emergency Department Indicator",
        "DimAdmissionContext",
        "ed_indicator_group",
        "INCLUDE",
        "Apply the approved category mapping.",
    ),
    (
        "Total Charges",
        "FactDischarge",
        "total_charges",
        "INCLUDE",
        "Parse numeric positive values and create validity flag.",
    ),
    (
        "Total Costs",
        "FactDischarge",
        "total_costs",
        "INCLUDE",
        "Parse numeric positive values and create validity flag.",
    ),
]

source_to_model_mapping = pd.DataFrame(
    source_mapping_rows,
    columns=[
        "source_field",
        "target_table",
        "target_column",
        "semantic_action",
        "transformation_requirement",
    ],
)

source_to_model_mapping

,source_field,target_table,target_column,semantic_action,transformation_requirement
0,Hospital Service Area,DimHospital,hospital_service_area,INCLUDE,Trim text and map nulls to Unknown.
1,Hospital County,DimHospital,hospital_county,INCLUDE,Trim text and map nulls to Unknown.
2,Operating Certificate Number,Staging,,STAGING_ONLY,Retained for source validation and lineage only. Not promoted to DimHospital because one Permanent Facility Id may h...
3,Permanent Facility Id,DimHospital,permanent_facility_id,INCLUDE,Retain natural identifier and derive hospital_key.
4,Facility Name,DimHospital,facility_name,INCLUDE,Trim text; do not use as the hospital key.
5,Age Group,DimPatientSegment,age_group,INCLUDE,Apply the approved category mapping.
6,Zip Code - 3 digits,DimGeography,patient_zip3,INCLUDE,Retain as text and preserve masked/unknown values.
7,Gender,DimPatientSegment,gender,INCLUDE,Trim and preserve released categories.
8,Race,DimPatientSegment,race,INCLUDE,Trim and preserve released categories.
9,Ethnicity,DimPatientSegment,ethnicity,INCLUDE,Trim and preserve released categories.


## 9. Relationship specification

In [12]:
relationship_pairs = [
    ("DimHospital", "hospital_key", "hospital_key"),
    ("DimDate", "date_key", "date_key"),
    ("DimService", "service_key", "service_key"),
    ("DimCaseMix", "case_mix_key", "case_mix_key"),
    ("DimDiagnosis", "diagnosis_key", "diagnosis_key"),
    ("DimProcedure", "procedure_key", "procedure_key"),
    (
        "DimPatientSegment",
        "patient_segment_key",
        "patient_segment_key",
    ),
    ("DimGeography", "geography_key", "geography_key"),
    ("DimPayer", "payer_key", "payer_key"),
    (
        "DimAdmissionContext",
        "admission_context_key",
        "admission_context_key",
    ),
]

relationship_rows = []

for dimension_table, dimension_key, fact_key in relationship_pairs:
    relationship_rows.append(
        {
            "relationship_name": (
                f"{dimension_table}[{dimension_key}] -> "
                f"FactDischarge[{fact_key}]"
            ),
            "from_table": dimension_table,
            "from_column": dimension_key,
            "to_table": "FactDischarge",
            "to_column": fact_key,
            "cardinality": "One-to-many (1:*)",
            "cross_filter_direction": "Single",
            "active": True,
            "unknown_member_key": 0,
            "validation_rule": (
                "Dimension key must be unique; every fact value must "
                "resolve to a dimension key or key 0."
            ),
        }
    )

relationship_specification = pd.DataFrame(relationship_rows)

relationship_specification

,relationship_name,from_table,from_column,to_table,to_column,cardinality,cross_filter_direction,active,unknown_member_key,validation_rule
0,DimHospital[hospital_key] -> FactDischarge[hospital_key],DimHospital,hospital_key,FactDischarge,hospital_key,One-to-many (1:*),Single,True,0,Dimension key must be unique; every fact value must resolve to a dimension key or key 0.
1,DimDate[date_key] -> FactDischarge[date_key],DimDate,date_key,FactDischarge,date_key,One-to-many (1:*),Single,True,0,Dimension key must be unique; every fact value must resolve to a dimension key or key 0.
2,DimService[service_key] -> FactDischarge[service_key],DimService,service_key,FactDischarge,service_key,One-to-many (1:*),Single,True,0,Dimension key must be unique; every fact value must resolve to a dimension key or key 0.
3,DimCaseMix[case_mix_key] -> FactDischarge[case_mix_key],DimCaseMix,case_mix_key,FactDischarge,case_mix_key,One-to-many (1:*),Single,True,0,Dimension key must be unique; every fact value must resolve to a dimension key or key 0.
4,DimDiagnosis[diagnosis_key] -> FactDischarge[diagnosis_key],DimDiagnosis,diagnosis_key,FactDischarge,diagnosis_key,One-to-many (1:*),Single,True,0,Dimension key must be unique; every fact value must resolve to a dimension key or key 0.
5,DimProcedure[procedure_key] -> FactDischarge[procedure_key],DimProcedure,procedure_key,FactDischarge,procedure_key,One-to-many (1:*),Single,True,0,Dimension key must be unique; every fact value must resolve to a dimension key or key 0.
6,DimPatientSegment[patient_segment_key] -> FactDischarge[patient_segment_key],DimPatientSegment,patient_segment_key,FactDischarge,patient_segment_key,One-to-many (1:*),Single,True,0,Dimension key must be unique; every fact value must resolve to a dimension key or key 0.
7,DimGeography[geography_key] -> FactDischarge[geography_key],DimGeography,geography_key,FactDischarge,geography_key,One-to-many (1:*),Single,True,0,Dimension key must be unique; every fact value must resolve to a dimension key or key 0.
8,DimPayer[payer_key] -> FactDischarge[payer_key],DimPayer,payer_key,FactDischarge,payer_key,One-to-many (1:*),Single,True,0,Dimension key must be unique; every fact value must resolve to a dimension key or key 0.
9,DimAdmissionContext[admission_context_key] -> FactDischarge[admission_context_key],DimAdmissionContext,admission_context_key,FactDischarge,admission_context_key,One-to-many (1:*),Single,True,0,Dimension key must be unique; every fact value must resolve to a dimension key or key 0.


### Relationship Interpretation

All descriptive dimensions filter `FactDischarge`.

No dimension-to-dimension, bidirectional, or many-to-many relationship is approved
for the initial model. The disconnected `Measures` table does not participate in any
relationship.

This specification defines expected cardinality. Actual key uniqueness and orphan-key
tests must be executed when the Power Query tables are physically constructed.

## 10. Metric-support matrix

In [13]:
def parse_required_fields(field_string):
    if pd.isna(field_string) or not str(field_string).strip():
        return set()

    return {
        field.strip()
        for field in str(field_string).split("|")
        if field.strip()
    }


included_source_fields = set(
    source_to_model_mapping.loc[
        source_to_model_mapping["semantic_action"] == "INCLUDE",
        "source_field",
    ]
)

metric_support_rows = []

for _, metric in metric_catalog.iterrows():
    required_fields = parse_required_fields(
        metric["required_fields"]
    )

    unsupported_fields = sorted(
        required_fields - included_source_fields
    )

    benchmark_deferred = (
        metric["metric_group"] == "Benchmarking"
    )
    upstream_status = str(metric["status"]).strip()

    if metric["metric_group"] == "Benchmarking":
        implementation_status = (
            "DESIGN_SUPPORTED_CALCULATION_DEFERRED"
        )
    elif upstream_status != "APPROVED":
        implementation_status = upstream_status
    else:
        implementation_status = "DESIGN_SUPPORTED"


    metric_support_rows.append(
        {
            "metric_id": metric["metric_id"],
            "metric_name": metric["metric_name"],
            "metric_group": metric["metric_group"],
            "required_fields": "|".join(sorted(required_fields)),
            "unsupported_fields": "|".join(unsupported_fields),
            "source_fields_supported": (
                len(unsupported_fields) == 0
            ),
            "implementation_status": implementation_status,
        }
    )


metric_support_matrix = pd.DataFrame(metric_support_rows)

metric_support_matrix

,metric_id,metric_name,metric_group,required_fields,unsupported_fields,source_fields_supported,implementation_status
0,discharge_count,Discharges,Volume,,,True,DESIGN_SUPPORTED
1,facility_count,Facilities,Volume,Permanent Facility Id,,True,DESIGN_SUPPORTED
2,total_los_days_lower_bound,Total LOS Days — Lower Bound,Length of Stay,Length of Stay,,True,DESIGN_SUPPORTED
3,average_los_lower_bound,Average LOS — Lower Bound,Length of Stay,Length of Stay,,True,DESIGN_SUPPORTED
4,median_los_lower_bound,Median LOS,Length of Stay,Length of Stay,,True,DESIGN_SUPPORTED
5,los_interquartile_range,LOS Interquartile Range,Length of Stay,Length of Stay,,True,DESIGN_SUPPORTED
6,p95_los_lower_bound,P95 LOS,Length of Stay,Length of Stay,,True,DESIGN_SUPPORTED
7,top_coded_los_count,Top-Coded LOS Discharges,Length of Stay,Length of Stay,,True,DESIGN_SUPPORTED
8,top_coded_los_rate,Top-Coded LOS Rate,Length of Stay,Length of Stay,,True,DESIGN_SUPPORTED
9,extended_stay_rate,Extended-Stay Rate,Length of Stay,Length of Stay,,True,PENDING_THRESHOLD


## 11. Key Policy

In [14]:
key_policy = pd.DataFrame(
    [
        {
            "policy_id": "INTEGER_SURROGATE_KEYS",
            "policy": (
                "All dimension relationships use whole-number "
                "surrogate keys."
            ),
            "implementation_stage": "Session 5",
        },
        {
            "policy_id": "UNKNOWN_KEY_ZERO",
            "policy": (
                "Key 0 represents an unresolved, missing, suppressed, "
                "or unavailable dimension member."
            ),
            "implementation_stage": "Session 5",
        },
        {
            "policy_id": "PRESERVE_FACT_ROWS",
            "policy": (
                "Missing dimension values map to key 0 rather than "
                "removing the discharge."
            ),
            "implementation_stage": "Session 5",
        },
        {
            "policy_id": "NATURAL_KEYS_VISIBLE",
            "policy": (
                "Released business identifiers remain available as "
                "descriptive dimension attributes."
            ),
            "implementation_stage": "Session 5",
        },
        {
            "policy_id": "TECHNICAL_KEYS_HIDDEN",
            "policy": (
                "Surrogate and foreign keys are hidden from report users."
            ),
            "implementation_stage": "Session 6",
        },
        {
            "policy_id": "NO_FACT_IDENTIFIER",
            "policy": (
                "The model does not invent a durable patient or "
                "discharge identifier."
            ),
            "implementation_stage": "Current design",
        },
        {
            "policy_id": "ANNUAL_DATE_KEY",
            "policy": (
                "The initial date key represents discharge year only."
            ),
            "implementation_stage": "Session 5",
        },
        {
            "policy_id": "FUTURE_MULTIYEAR_REVIEW",
            "policy": (
                "Before appending additional years, test dimension "
                "attribute stability and key reuse across years."
            ),
            "implementation_stage": "Future multi-year build",
        },
    ]
)

key_policy

,policy_id,policy,implementation_stage
0,INTEGER_SURROGATE_KEYS,All dimension relationships use whole-number surrogate keys.,Session 5
1,UNKNOWN_KEY_ZERO,"Key 0 represents an unresolved, missing, suppressed, or unavailable dimension member.",Session 5
2,PRESERVE_FACT_ROWS,Missing dimension values map to key 0 rather than removing the discharge.,Session 5
3,NATURAL_KEYS_VISIBLE,Released business identifiers remain available as descriptive dimension attributes.,Session 5
4,TECHNICAL_KEYS_HIDDEN,Surrogate and foreign keys are hidden from report users.,Session 6
5,NO_FACT_IDENTIFIER,The model does not invent a durable patient or discharge identifier.,Current design
6,ANNUAL_DATE_KEY,The initial date key represents discharge year only.,Session 5
7,FUTURE_MULTIYEAR_REVIEW,"Before appending additional years, test dimension attribute stability and key reuse across years.",Future multi-year build


## 12. Schema validation

In [15]:
table_names = set(table_specification["table_name"])

model_column_pairs = set(
    zip(
        column_specification["table_name"],
        column_specification["column_name"],
    )
)

relationship_endpoints_exist = all(
    (
        row["from_table"],
        row["from_column"],
    ) in model_column_pairs
    and (
        row["to_table"],
        row["to_column"],
    ) in model_column_pairs
    for _, row in relationship_specification.iterrows()
)

fact_foreign_key_columns = set(
    column_specification.loc[
        (
            column_specification["table_name"].eq("FactDischarge")
            & column_specification["column_role"].eq("Foreign key")
        ),
        "column_name",
    ]
)

related_fact_columns = set(
    relationship_specification["to_column"]
)

dimension_tables = set(
    table_specification.loc[
        table_specification["table_type"].eq("Dimension"),
        "table_name",
    ]
)

related_dimensions = set(
    relationship_specification["from_table"]
)

mapped_source_fields = set(
    source_to_model_mapping["source_field"]
)

staging_only_rows = source_to_model_mapping.loc[
    source_to_model_mapping["semantic_action"].eq("STAGING_ONLY")
]
required_metric_model_columns = {
    "extended_stay_rate": {
        ("FactDischarge", "los_days_lower_bound"),
    },
    "disposition_share": {
        ("DimAdmissionContext", "disposition_group"),
    },
    "estimated_cost_per_inpatient_day": {
        ("FactDischarge", "total_costs"),
        ("FactDischarge", "los_days_lower_bound"),
    },
    "home_discharge_rate": {
        ("DimAdmissionContext", "disposition_group"),
    },
    "in_hospital_mortality_rate": {
        ("DimAdmissionContext", "disposition_group"),
    },
    "average_apr_severity_index": {
        ("DimCaseMix", "apr_severity_code"),
    },
    "peer_expected_los_days": {
        ("FactDischarge", "peer_expected_los_days"),
        ("FactDischarge", "los_peer_comparison_n"),
        ("FactDischarge", "los_peer_benchmark_level"),
    },
    "los_actual_to_peer_expected_ratio": {
        ("FactDischarge", "los_days_lower_bound"),
        ("FactDischarge", "peer_expected_los_days"),
    },
    "excess_los_days_lower_bound": {
        ("FactDischarge", "los_days_lower_bound"),
        ("FactDischarge", "peer_expected_los_days"),
    },
    "peer_expected_estimated_cost": {
        ("FactDischarge", "peer_expected_estimated_cost"),
        ("FactDischarge", "cost_peer_comparison_n"),
        ("FactDischarge", "cost_peer_benchmark_level"),
    },
    "estimated_cost_actual_to_peer_expected_ratio": {
        ("FactDischarge", "total_costs"),
        ("FactDischarge", "peer_expected_estimated_cost"),
    },
    "excess_estimated_cost": {
        ("FactDischarge", "total_costs"),
        ("FactDischarge", "peer_expected_estimated_cost"),
    },
}

catalog_metric_ids = set(metric_catalog["metric_id"])

metric_rules_without_catalog_metric = sorted(
    set(required_metric_model_columns) - catalog_metric_ids
)

missing_metric_model_columns = {}

for metric_id, required_columns in required_metric_model_columns.items():
    missing_columns = sorted(
        required_columns - model_column_pairs
    )

    if missing_columns:
        missing_metric_model_columns[metric_id] = missing_columns

extended_stay_status = metric_support_matrix.loc[
    metric_support_matrix["metric_id"].eq("extended_stay_rate"),
    "implementation_status",
]

extended_stay_status_preserved = (
    len(extended_stay_status) == 1
    and extended_stay_status.iloc[0] == "PENDING_THRESHOLD"
)

dimservice_spec = table_specification.loc[
    table_specification["table_name"].eq("DimService")
]

dimservice_composite_key_valid = (
    len(dimservice_spec) == 1
    and dimservice_spec.iloc[0]["natural_key"]
    == "apr_drg_code|apr_mdc_code"
)

dimservice_key_components = set(
    column_specification.loc[
        (
            column_specification["table_name"].eq("DimService")
            & column_specification["column_role"].eq(
                "Natural key component"
            )
        ),
        "column_name",
    ]
)

expected_dimservice_key_components = {
    "apr_drg_code",
    "apr_mdc_code",
}

dimservice_key_components_valid = (
    dimservice_key_components
    == expected_dimservice_key_components
)

validation_rows = [
    {
        "validation_test": "Table names are unique",
        "passed": table_specification["table_name"].is_unique,
        "details": "",
    },
    {
        "validation_test": "Model columns are unique within tables",
        "passed": ~column_specification.duplicated(
            ["table_name", "column_name"]
        ).any(),
        "details": "",
    },
    {
        "validation_test": "All source fields are explicitly classified",
        "passed": mapped_source_fields == available_columns,
        "details": (
            "Unmapped: "
            + "|".join(sorted(available_columns - mapped_source_fields))
            + "; Unexpected: "
            + "|".join(sorted(mapped_source_fields - available_columns))
        ),
    },

 {
    "validation_test": (
        "Metric-specific model rules reference catalog metrics"
    ),
    "passed": not metric_rules_without_catalog_metric,
    "details": "|".join(metric_rules_without_catalog_metric),
},
{
    "validation_test": (
        "Required model columns exist"
    ),
    "passed": not missing_metric_model_columns,
    "details": str(missing_metric_model_columns),
},
    {
        "validation_test": "Source-field mappings are unique",
        "passed": source_to_model_mapping["source_field"].is_unique,
        "details": "",
    },
    {
        "validation_test": "Semantic actions are valid",
        "passed": set(
            source_to_model_mapping["semantic_action"]
        ).issubset({"INCLUDE", "STAGING_ONLY"}),
        "details": "",
    },
    {
    "validation_test": (
        "DimService uses validated composite natural key"
    ),
    "passed": dimservice_composite_key_valid,
    "details": (
        ""
        if dimservice_composite_key_valid
        else "|".join(
            dimservice_spec["natural_key"].astype(str)
        )
    ),
    },
    {
        "validation_test": (
            "DimService composite-key columns are explicitly identified"
        ),
        "passed": dimservice_key_components_valid,
        "details": "|".join(
            sorted(dimservice_key_components)
        ),
    },
    {
        "validation_test": "Staging-only exclusions are documented",
        "passed": staging_only_rows[
            "transformation_requirement"
        ].fillna("").str.strip().ne("").all(),
        "details": "",
    },
    {
        "validation_test": "All metric source dependencies are supported",
        "passed": metric_support_matrix[
            "source_fields_supported"
        ].all(),
        "details": "|".join(
            metric_support_matrix.loc[
                ~metric_support_matrix[
                    "source_fields_supported"
                ],
                "metric_id",
            ]
        ),
    },
    {
        "validation_test": "Relationship endpoints exist",
        "passed": relationship_endpoints_exist,
        "details": "",
    },
    {
        "validation_test": "Every fact foreign key has one relationship",
        "passed": fact_foreign_key_columns == related_fact_columns,
        "details": "",
    },
    {
        "validation_test": "Every dimension relates directly to the fact",
        "passed": dimension_tables == related_dimensions,
        "details": "",
    },
    {
        "validation_test": "Relationships target FactDischarge",
        "passed": relationship_specification[
            "to_table"
        ].eq("FactDischarge").all(),
        "details": "",
    },
    {
        "validation_test": "Relationships are one-to-many",
        "passed": relationship_specification[
            "cardinality"
        ].eq("One-to-many (1:*)").all(),
        "details": "",
    },
    {
        "validation_test": "Relationships use single-direction filtering",
        "passed": relationship_specification[
            "cross_filter_direction"
        ].eq("Single").all(),
        "details": "",
    },
    {
        "validation_test": "All relationships are active",
        "passed": relationship_specification["active"].all(),
        "details": "",
    },
    {
        "validation_test": "Unknown-member key is consistently zero",
        "passed": relationship_specification[
            "unknown_member_key"
        ].eq(0).all(),
        "details": "",
    },
    {
        "validation_test": "All relationship tables are defined",
        "passed": (
            set(relationship_specification["from_table"])
            | set(relationship_specification["to_table"])
        ).issubset(table_names),
        "details": "",
    },
    {
    "validation_test": (
        "Extended-Stay Rate pending status is preserved"
    ),
    "passed": extended_stay_status_preserved,
    "details": (
        ""
        if extended_stay_status_preserved
        else "|".join(extended_stay_status.astype(str))
    ),
    },
]

schema_validation_results = pd.DataFrame(validation_rows)

display(schema_validation_results)

assert schema_validation_results["passed"].all(), (
    "The star-schema specification failed one or more validation tests."
)

,validation_test,passed,details
0,Table names are unique,True,
1,Model columns are unique within tables,True,
2,All source fields are explicitly classified,True,Unmapped: ; Unexpected:
3,Metric-specific model rules reference catalog metrics,True,
4,Required model columns exist,True,{}
5,Source-field mappings are unique,True,
6,Semantic actions are valid,True,
7,DimService uses validated composite natural key,True,
8,DimService composite-key columns are explicitly identified,True,apr_drg_code|apr_mdc_code
9,Staging-only exclusions are documented,True,


## 13. Schema Diagram

## Logical Star Schema

```mermaid
flowchart TD
    H["DimHospital"] --> F["FactDischarge"]
    D["DimDate"] --> F
    S["DimService"] --> F
    C["DimCaseMix"] --> F
    DX["DimDiagnosis"] --> F
    PX["DimProcedure"] --> F
    PS["DimPatientSegment"] --> F
    G["DimGeography"] --> F
    P["DimPayer"] --> F
    A["DimAdmissionContext"] --> F
    M["Measures — Disconnected"]
```

## 14. Export specifications


In [16]:
export_tables = {
    "design_standards.csv": design_standards,
    "table_specification.csv": table_specification,
    "column_specification.csv": column_specification,
    "source_to_model_mapping.csv": source_to_model_mapping,
    "relationship_specification.csv": relationship_specification,
    "key_policy.csv": key_policy,
    "metric_support_matrix.csv": metric_support_matrix,
    "schema_validation_results.csv": schema_validation_results,
}

export_manifest_rows = []

for file_name, dataframe in export_tables.items():
    output_path = OUTPUT_DIR / file_name

    dataframe.to_csv(output_path, index=False)

    assert output_path.exists()
    assert output_path.stat().st_size > 0

    export_manifest_rows.append(
        {
            "file_name": file_name,
            "row_count": len(dataframe),
            "output_path": str(
                output_path.relative_to(PROJECT_ROOT)
            ),
        }
    )

export_manifest = pd.DataFrame(export_manifest_rows)

export_manifest_path = OUTPUT_DIR / "export_manifest.csv"

export_manifest.to_csv(
    export_manifest_path,
    index=False,
)

assert export_manifest_path.exists()
assert export_manifest_path.stat().st_size > 0

display(export_manifest)

,file_name,row_count,output_path
0,design_standards.csv,14,outputs\star_schema\design_standards.csv
1,table_specification.csv,12,outputs\star_schema\table_specification.csv
2,column_specification.csv,63,outputs\star_schema\column_specification.csv
3,source_to_model_mapping.csv,33,outputs\star_schema\source_to_model_mapping.csv
4,relationship_specification.csv,10,outputs\star_schema\relationship_specification.csv
5,key_policy.csv,8,outputs\star_schema\key_policy.csv
6,metric_support_matrix.csv,32,outputs\star_schema\metric_support_matrix.csv
7,schema_validation_results.csv,21,outputs\star_schema\schema_validation_results.csv


## 15. Generate docs/star_schema.md

In [17]:
def dataframe_to_markdown(dataframe, columns):
    selected = dataframe.loc[:, columns].fillna("").astype(str)

    def escape_value(value):
        return (
            value
            .replace("|", r"\|")
            .replace("\n", " ")
        )

    header = "| " + " | ".join(columns) + " |"
    separator = "| " + " | ".join(
        ["---"] * len(columns)
    ) + " |"

    rows = [
        "| "
        + " | ".join(
            escape_value(value)
            for value in row
        )
        + " |"
        for row in selected.itertuples(
            index=False,
            name=None,
        )
    ]

    return "\n".join(
        [header, separator, *rows]
    )


design_standard_markdown = dataframe_to_markdown(
    design_standards,
    [
        "standard_id",
        "decision",
        "rationale",
    ],
)

table_markdown = dataframe_to_markdown(
    table_specification,
    [
        "table_name",
        "table_type",
        "grain",
        "primary_key",
        "natural_key",
        "business_role",
        "important_caveat",
    ],
)

column_markdown = dataframe_to_markdown(
    column_specification,
    [
        "table_name",
        "column_name",
        "data_type",
        "column_role",
        "source_fields",
        "status",
    ],
)

relationship_markdown = dataframe_to_markdown(
    relationship_specification,
    [
        "from_table",
        "from_column",
        "to_table",
        "to_column",
        "cardinality",
        "cross_filter_direction",
    ],
)

source_mapping_markdown = dataframe_to_markdown(
    source_to_model_mapping,
    [
        "source_field",
        "target_table",
        "target_column",
        "semantic_action",
        "transformation_requirement",
    ],
)

metric_support_markdown = dataframe_to_markdown(
    metric_support_matrix,
    [
        "metric_id",
        "metric_name",
        "source_fields_supported",
        "implementation_status",
    ],
)

key_policy_markdown = dataframe_to_markdown(
    key_policy,
    [
        "policy_id",
        "policy",
        "implementation_stage",
    ],
)

validation_markdown = dataframe_to_markdown(
    schema_validation_results,
    [
        "validation_test",
        "passed",
        "details",
    ],
)

mermaid_diagram = """```mermaid
flowchart TD
    H["DimHospital"] --> F["FactDischarge"]
    D["DimDate"] --> F
    S["DimService"] --> F
    C["DimCaseMix"] --> F
    DX["DimDiagnosis"] --> F
    PX["DimProcedure"] --> F
    PS["DimPatientSegment"] --> F
    G["DimGeography"] --> F
    P["DimPayer"] --> F
    A["DimAdmissionContext"] --> F
    M["Measures — Disconnected"]
```"""

schema_document_sections = [
    "# Star Schema — Hospital Operations & Cost Efficiency",
    "",
    "## Purpose",
    "",
    (
        "This document defines the approved logical star schema "
        "for the initial SPARCS hospital-operations semantic model."
    ),
    "",
    "## Fact Grain",
    "",
    (
        "`FactDischarge` contains one row per released "
        "inpatient discharge."
    ),
    "",
    (
        "The source does not contain a reliable patient or durable "
        "discharge identifier. Discharge counts must not be "
        "interpreted as unique-patient counts."
    ),
    "",
    "## Logical Model",
    "",
    mermaid_diagram,
    "",
    "## Design Standards",
    "",
    design_standard_markdown,
    "",
    "## Table Specifications",
    "",
    table_markdown,
    "",
    "## Column Specifications",
    "",
    column_markdown,
    "",
    "## Relationships",
    "",
    relationship_markdown,
    "",
    "## Source-to-Model Mapping",
    "",
    source_mapping_markdown,
    "",
    "## Metric Support",
    "",
    metric_support_markdown,
    "",
    "## Key Policy",
    "",
    key_policy_markdown,
    "",
    "## Validation Results",
    "",
    validation_markdown,
    "",
    "## Important Limitations",
    "",
    (
        "- The fact table represents discharges rather than "
        "unique patients."
    ),
    (
        "- Monthly analysis is unavailable because the source "
        "contains discharge year only."
    ),
    (
        "- Peer benchmarks remain descriptive and are not formal "
        "clinical risk adjustment."
    ),
    (
        "- Physical key uniqueness, orphan detection, and row "
        "reconciliation remain deferred until table construction."
    ),
    "",
]

schema_document = "\n".join(
    schema_document_sections
)

STAR_SCHEMA_DOCUMENT_PATH = (
    DOCS_DIR / "star_schema.md"
)

STAR_SCHEMA_DOCUMENT_PATH.write_text(
    schema_document,
    encoding="utf-8",
)

assert STAR_SCHEMA_DOCUMENT_PATH.exists()
assert STAR_SCHEMA_DOCUMENT_PATH.stat().st_size > 0

document_manifest_row = pd.DataFrame(
    [
        {
            "file_name": STAR_SCHEMA_DOCUMENT_PATH.name,
            "row_count": pd.NA,
            "output_path": str(
                STAR_SCHEMA_DOCUMENT_PATH.relative_to(
                    PROJECT_ROOT
                )
            ),
        }
    ]
)

final_export_manifest = pd.concat(
    [
        export_manifest,
        document_manifest_row,
    ],
    ignore_index=True,
)

final_export_manifest.to_csv(
    OUTPUT_DIR / "export_manifest.csv",
    index=False,
)

for relative_path in final_export_manifest["output_path"]:
    assert (PROJECT_ROOT / relative_path).exists()

display(final_export_manifest)

print(
    "Star-schema documentation created:",
    STAR_SCHEMA_DOCUMENT_PATH.relative_to(
        PROJECT_ROOT
    ),
)

,file_name,row_count,output_path
0,design_standards.csv,14,outputs\star_schema\design_standards.csv
1,table_specification.csv,12,outputs\star_schema\table_specification.csv
2,column_specification.csv,63,outputs\star_schema\column_specification.csv
3,source_to_model_mapping.csv,33,outputs\star_schema\source_to_model_mapping.csv
4,relationship_specification.csv,10,outputs\star_schema\relationship_specification.csv
5,key_policy.csv,8,outputs\star_schema\key_policy.csv
6,metric_support_matrix.csv,32,outputs\star_schema\metric_support_matrix.csv
7,schema_validation_results.csv,21,outputs\star_schema\schema_validation_results.csv
8,star_schema.md,<NA>,docs\star_schema.md


Star-schema documentation created: docs\star_schema.md


# Final Notebook Summary

## Work Completed

- Loaded and validated Notebook 01 and the updated Notebook 02 specifications.
- Confirmed that required project metrics and primary benchmarks are present.
- Defined the one-discharge-per-row fact-table grain.
- Defined fact, dimension, and disconnected measure tables.
- Classified every audited source field as included or staging-only.
- Defined whole-number surrogate-key and unknown-member policies.
- Defined active one-to-many, single-direction relationships.
- Confirmed that required metric dependencies are represented in the model.
- Defined deferred storage requirements for the approved LOS and estimated-cost peer benchmarks.
- Exported machine-readable schema specifications.
- Generated `docs/star_schema.md`.

## Key Design Decisions

1. `FactDischarge` contains one row per released discharge.
2. The model does not create a patient identifier.
3. All descriptive dimensions relate directly to the fact table.
4. Relationships are one-to-many and single-directional.
5. Key 0 represents unresolved or unavailable dimension values.
6. The initial time dimension supports annual analysis only.
7. Primary payer is modeled; secondary and tertiary payer roles remain in staging.
8. Patient ZIP3 is separated from hospital geography.
9. Patient disposition is available for descriptive analysis but not as a primary benchmark input.
10. APR severity is stored numerically from 1 through 4 after validation.
11. `DimService` uses `APR-DRG Code × APR MDC Code` as its composite natural key because APR-DRG alone does not uniquely determine MDC in the validated 2023 source.
12. LOS and estimated-cost peer expectations exclude the focal facility.
13. Primary LOS and estimated-cost peer groups use APR-DRG and severity, with APR-DRG alone as the documented fallback.
14. Peer expectations remain deferred until benchmark computation.
15. Extended-Stay Rate remains pending until its operational LOS threshold is approved.

## Remaining Physical Validations

The following must be tested when the Power Query tables are constructed:

- Dimension-key uniqueness
- Physical natural-key-to-description consistency for all dimensions
- Orphan foreign keys
- Fact-row reconciliation
- Unknown-member assignment
- Data-type conversion
- Financial and LOS validity flags
- Model memory usage
- Actual Power BI relationship cardinality

## Scope Boundary

This notebook does not build Power Query transformations, DAX measures,
dashboard pages, hospital rankings, predictive models, or Fabric pipelines.